# Day 7: Testing Data & ML Code with Pytest

## Why Test Data/ML Code?
A silent bug in a cleaning function or feature pipeline can quietly corrupt every model trained after it - tests catch that before it ships.

In [ ]:
# pip install pytest

# project layout:
# my_project/
#     cleaning.py
#     test_cleaning.py   <- pytest auto-discovers files named test_*.py

## Writing Your First Test
A test is just a function starting with test_ that uses assert to check an expectation.

In [ ]:
# cleaning.py
def clean_price(value):
    """Remove currency symbols/commas and convert to float."""
    return float(str(value).replace("$", "").replace(",", ""))

# test_cleaning.py
from cleaning import clean_price

def test_clean_price_removes_symbols():
    assert clean_price("$1,200.50") == 1200.50

def test_clean_price_handles_plain_number():
    assert clean_price(99) == 99.0

## Testing a Data Cleaning Function on a Small DataFrame
Build a tiny, known DataFrame and assert the function does exactly what you expect on it.

In [ ]:
# test_cleaning_df.py
import pandas as pd

def drop_duplicate_emails(df):
    return df.drop_duplicates(subset="email")

def test_drop_duplicate_emails():
    df = pd.DataFrame({
        "email": ["a@x.com", "a@x.com", "b@x.com"],
        "name": ["Ann", "Ann", "Bob"],
    })
    result = drop_duplicate_emails(df)
    assert len(result) == 2
    assert result["email"].is_unique

## Testing a Model's Output Shape & Range
For ML code, assert on structural properties (shape, value range, no NaNs) rather than exact numbers, since training can vary slightly.

In [ ]:
# test_model.py
import numpy as np
from sklearn.linear_model import LogisticRegression

def test_predictions_are_valid_probabilities():
    X = np.array([[0, 0], [1, 1], [1, 0], [0, 1]])
    y = np.array([0, 1, 1, 0])
    model = LogisticRegression().fit(X, y)

    probs = model.predict_proba(X)
    assert probs.shape == (4, 2)
    assert np.all((probs >= 0) & (probs <= 1))
    assert np.allclose(probs.sum(axis=1), 1.0)

## Fixtures — Reusable Test Data
A fixture builds shared setup once (e.g. a sample DataFrame) and hands it to every test that asks for it.

In [ ]:
# test_with_fixture.py
import pytest
import pandas as pd

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "age": [25, -5, 40, 200],
        "salary": [50000, 60000, None, 80000],
    })

def test_no_negative_ages_after_cleaning(sample_df):
    cleaned = sample_df[sample_df["age"].between(0, 120)]
    assert (cleaned["age"] >= 0).all()
    assert (cleaned["age"] <= 120).all()

## Running the Test Suite
One command runs every test file and gives a pass/fail report - this is what a CI pipeline runs automatically on every push.

In [ ]:
# terminal commands:
# pytest                      # run all tests in the project
# pytest -v                   # verbose: show each test name
# pytest test_cleaning.py     # run just one file
# pytest -k "price"           # run only tests matching "price"